In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import functional as F_t
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import kagglehub

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

train_root = f"{path}/Training"
test_root = f"{path}/Testing"

In [ ]:
print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
device

In [ ]:
class ResizeWithPad():
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        # resize to desired size first
        w, h = img.size
        scale = self.size / max(w, h)
        new_w, new_h = int(scale * w), int(scale * h)
        img = F_t.resize(img, (new_h, new_w))

        pad_w = self.size - new_w
        pad_h = self.size - new_h

        # padding for left top right bottom respectively
        padding = (pad_w // 2, pad_h // 2, pad_w - pad_w // 2, pad_h - pad_h // 2)

        # the 'fill=0' ensures the padding is of the colour black
        img = F_t.pad(img, padding, fill=0)

        return img

In [ ]:
img_size = 224

transformTrain = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                                     transforms.RandomHorizontalFlip(),
                                     transforms.RandomRotation(10),
                                     ResizeWithPad(img_size),
                                     transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.5], std=[0.5])
                                    ])

transformEval = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                                     ResizeWithPad(img_size),
                                     transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.5], std=[0.5])
                                    ])

In [ ]:
full_train_aug = datasets.ImageFolder(
    root=train_root,
    transform=transformTrain
)

full_train_eval = datasets.ImageFolder(
    root=train_root,
    transform=transformEval
)

targets = [label for _, label in full_train_aug.samples]

train_index, val_index = train_test_split(
    np.arange(len(targets)),
    test_size=0.15,
    stratify=targets,
    random_state=67
)

train_data = Subset(full_train_aug, train_index) # has augmentation
val_data = Subset(full_train_eval, val_index) # no augmentation
test_data = datasets.ImageFolder(
    root=test_root,
    transform=transformEval
)

In [ ]:
# loaded data for processing

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# CNN model class
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # block 1: 1 -> 32 channels
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)


        # block 2: 32 -> 64 channels
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)

        # block 3: 64 -> 128 channels
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.conv6 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.fc1 = nn.Linear(128, 128)
        self.dropout1 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, X):
        X = F.relu(self.bn1(self.conv1(X))) # first convolution layer (then ReLU)
        X = F.relu(self.bn2(self.conv2(X))) # second convolution layer (then ReLU)

        X = self.pool(X) # from 224 -> 112

        X = F.relu(self.bn3(self.conv3(X))) # third convolution layer (then ReLU)
        X = F.relu(self.bn4(self.conv4(X))) # fourth convolution layer (then ReLU)

        X = self.pool(X) # from 112 -> 56


        X = F.relu(self.bn5(self.conv5(X))) # fifth convolution layer (then ReLU)
        X = F.relu(self.bn6(self.conv6(X))) # sixth convolution layer (then ReLU)

        X = self.pool(X) # from 56 -> 28

        X = self.global_pool(X) # goes from 28x28 feature maps to 1x1
        X = X.view(X.size(0), -1) # flatten to (batch, 128)

        X = F.relu(self.fc1(X)) # first fully connected layer (then ReLU)
        X = self.dropout1(X) # first dropout
        X = self.fc2(X) # second fully connected layer

        return X


In [ ]:
# setting seed for debugging purposes
torch.manual_seed(42)
model = ConvolutionalNetwork()
model = model.to(device)

In [ ]:
class_weights = torch.tensor([2.0, 1.0, 1.0, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser, mode="min", factor=0.5, patience=3
)

In [ ]:
# create variables for training
epochs = 50
train_losses_list, train_accuracy_list = [], []
val_losses_list, val_accuracy_list = [], []

best_val_loss = float("inf")

for epoch in range(epochs):

    # training
    model.train()
    train_correct, epoch_train_loss = 0, 0

    model.train()
    for X_train, y_train in train_loader:
        X_train, y_train = X_train.to(device), y_train.to(device)

        y_pred = model(X_train) # X_train is basically the image
        loss = criterion(y_pred, y_train)

        # update the parameters
        optimiser.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()

        predicted = torch.max(y_pred.data, 1)[1]
        batch_correct = (predicted == y_train).sum().item() # the '==' is pairwise
        train_correct += batch_correct # add to the total number that are correct
        epoch_train_loss += loss.item()

    # we normalise here by dividing loss and accuracy by len of train_loader and train_data respectively
    # to compare with val_losses and val_accuracy
    train_losses_list.append(epoch_train_loss/len(train_loader))
    train_accuracy_list.append(train_correct/len(train_data)) # train_accuracy is cumulative


    # testing
    model.eval()
    val_correct, epoch_val_loss = 0, 0

    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            y_pred = model(X_val)
            loss = criterion(y_pred, y_val) # not needed in the for loop since we aren't training on it

            predicted = torch.max(y_pred.data, 1)[1]
            val_batch_correct = (predicted == y_val).sum().item()
            val_correct += val_batch_correct
            epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss/len(val_loader) # divide by number of batches
        val_losses_list.append(avg_val_loss)
        val_accuracy_list.append(val_correct/len(val_data)) # cumulative

        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'my-mri-model.pt')

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train loss: {train_losses_list[-1]:.4f}, accuracy: {train_accuracy_list[-1]:.4f} | "
          f"Validation loss: {avg_val_loss:.4f}, accuracy: {val_accuracy_list[-1]:.4f}")

In [ ]:
plt.plot(train_losses_list, label="Training Loss")
plt.plot(val_losses_list, label="Validation Loss")
plt.legend()
plt.show()

plt.plot(train_accuracy_list, label="Training Accuracy")
plt.plot(val_accuracy_list, label="Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
resultant_model = ConvolutionalNetwork()
resultant_model = resultant_model.to(device)
resultant_model.load_state_dict(torch.load('my-mri-model.pt'))
resultant_model.eval() # disables dropout

test_correct = 0
with torch.no_grad(): # disables backpropagation results being stored, so efficient

    for X_test, y_test in test_loader:
        X_test, y_test = X_test.to(device), y_test.to(device)
        y_pred = resultant_model(X_test)
        predicted = torch.max(y_pred, 1)[1]
        test_correct += (predicted == y_test).sum().item()

test_accuracy = test_correct / len(test_data)
print(f"Final test accuracy: {test_accuracy:.4f} ({test_correct}/{len(test_data)})")

In [ ]:
tumour_dict = {
    0 : "glioma",
    1 : "meningioma",
    2 : "notumour",
    3 : "pituitary"
}

# for testing specific sample data
test_sample_number = 1000
test_sample_classification = test_data[test_sample_number][1]
test_sample_data = test_data[test_sample_number][0]

plt.imshow(test_sample_data.squeeze(), cmap="grey")
plt.show()

with torch.no_grad():
    test_sample_data = test_sample_data.to(device)
    logits = resultant_model(test_sample_data.unsqueeze(0)) # the unsqueeze adds a dimension for the batch that is expected by the model
    predicted = logits.argmax().item() # logits are unbounded scores, the model's confidence

    print(f'Predicted: {tumour_dict[predicted]}, Actual: {tumour_dict[test_sample_classification]}')

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
 
# collect every prediction and true label from the whole test set
all_preds = []
all_labels = []
 
resultant_model.eval()
with torch.no_grad():
    for X_test, y_test in test_loader:
        X_test, y_test = X_test.to(device), y_test.to(device)
        y_pred = resultant_model(X_test)
        predicted = torch.max(y_pred, 1)[1]
 
        # .cpu() moves the tensor off gpu before converting to plain list since sklearn works with numpy/cpu data not gpu tensors
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_test.cpu().numpy())
 
class_names = [tumour_dict[i] for i in range(len(tumour_dict))]
 
# confusion matrix: rows represents actual class, columns represents predicted class,
# the diagonal represents correct predictions, anything off the diagonal is an error
cm = confusion_matrix(all_labels, all_preds)
 
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Test Set')
plt.show()
 
# for each class we produce the precision, recall, f1 to say if drop is spread evenly across
# all 4 classes or concentrated in certain classes
print(classification_report(all_labels, all_preds, target_names=class_names))